# Latihan Desain Sistem Kontrol — Kelompok 3
## G(s) = 8 / [(s+1)(s+2)(s+6)]

**Spesifikasi kinerja:**
- Overshoot (OS) ≤ 20%
- Settling time (ts) ≤ 4 s (kriteria 2%)
- Gain Margin (GM) ≥ 10 dB

**Tugas:**
1. Analisis plant (open-loop)
2. Desain PID menggunakan metode Ziegler-Nichols
3. Desain Kompensator Lead menggunakan metode Bode
4. Bandingkan kedua pendekatan

## 0. Import Library

In [ ]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch

# Matplotlib styling
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
print('Library berhasil di-import.')

## 1. Definisi Plant

$$G(s) = \frac{8}{(s+1)(s+2)(s+6)} = \frac{8}{s^3 + 9s^2 + 20s + 12}$$

Ekspansi denominator: $(s+1)(s+2)(s+6)$
- $(s+1)(s+2) = s^2 + 3s + 2$
- $(s^2+3s+2)(s+6) = s^3 + 6s^2 + 3s^2 + 18s + 2s + 12 = s^3 + 9s^2 + 20s + 12$ ✓

In [ ]:
# ---- Plant definition ----
num_G = [8]                     # pembilang: 8
den_G = [1, 9, 20, 12]          # penyebut: s^3 + 9s^2 + 20s + 12

G = signal.TransferFunction(num_G, den_G)

# ---- Analisis poles dan zeros ----
poles = np.roots(den_G)
zeros = np.roots(num_G) if len(num_G) > 1 else []

# ---- Tipe sistem (jumlah pole di origin) ----
n_integrators = int(np.sum(np.isclose(np.abs(poles), 0, atol=1e-6)))

print('=' * 50)
print(f'  G(s) = {num_G[0]} / (s³ + 9s² + 20s + 12)')
print('=' * 50)
print(f'  Poles  : {poles}')
print(f'  Zeros  : {"Tidak ada" if len(zeros)==0 else zeros}')
print(f'  Tipe   : {n_integrators} (Tipe 0 — tidak ada integrator)')
print(f'  Orde   : {len(poles)}')
print()
print('  DC Gain (K_position) = G(0) =', num_G[0]/den_G[-1])
print(f'  → steady-state error utk input step = 1/(1+Kp) = {1/(1+num_G[0]/den_G[-1]):.4f}')

## 2. Analisis Open-Loop: Bode Diagram

Sebelum mendesain kontroler, kita perlu mengetahui:
- **Phase Margin (PM)** — ukuran kestabilan pada gain crossover frequency (ωcg)
- **Gain Margin (GM)** — ukuran kestabilan pada phase crossover frequency (ωcp)

Kriteria kestabilan Bode: sistem closed-loop stabil jika PM > 0° dan GM > 0 dB.

In [ ]:
def compute_margins(sys, w_range=None):
    """Hitung Phase Margin (PM) dan Gain Margin (GM) dari data Bode."""
    if w_range is None:
        w_range = np.logspace(-3, 5, 10000)
    w, mag_dB, phase_deg = signal.bode(sys, w=w_range)

    PM, wcg, GM, wcp = None, None, None, None

    # Phase Margin: cari ωcg di mana |G(jω)| = 1 (0 dB)
    cross0 = np.where(np.diff(np.sign(mag_dB)))[0]
    for idx in cross0:
        t = -mag_dB[idx] / (mag_dB[idx+1] - mag_dB[idx])
        wc = np.exp(np.log(w[idx]) + t*(np.log(w[idx+1]) - np.log(w[idx])))
        ph_at_wc = phase_deg[idx] + t*(phase_deg[idx+1] - phase_deg[idx])
        PM = 180 + ph_at_wc
        wcg = wc
        break   # ambil crossover pertama

    # Gain Margin: cari ωcp di mana ∠G(jω) = -180°
    cross180 = np.where(np.diff(np.sign(phase_deg + 180)))[0]
    for idx in cross180:
        if phase_deg[idx] < -30:    # pastikan crossing dari atas
            t = -(phase_deg[idx] + 180) / (phase_deg[idx+1] - phase_deg[idx])
            wc = np.exp(np.log(w[idx]) + t*(np.log(w[idx+1]) - np.log(w[idx])))
            mag_at_wc = mag_dB[idx] + t*(mag_dB[idx+1] - mag_dB[idx])
            GM = -mag_at_wc        # GM = -|G(jωcp)|_dB
            wcp = wc
            break

    return PM, GM, wcg, wcp


# Hitung margin
PM, GM, wcg, wcp = compute_margins(G)

print('Margin Stabilitas — G(s) Open-Loop')
print('-' * 40)
print(f'  Phase Margin  PM  = {PM:.2f}°   (di ωcg = {wcg:.3f} rad/s)')
print(f'  Gain Margin   GM  = {GM:.2f} dB  (di ωcp = {wcp:.3f} rad/s)')
print()
print('  ✓ PM > 0° dan GM > 0 dB → sistem CL dengan unity feedback STABIL')
print(f'  ⚠ GM = {GM:.2f} dB < 10 dB (spec). Perlu perbaikan margin!')

# Plot Bode
w = np.logspace(-2, 3, 1000)
w_out, mag_dB, phase_deg = signal.bode(G, w=w)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
fig.suptitle('Bode Diagram — G(s) = 8 / [(s+1)(s+2)(s+6)]', fontsize=13)

# Magnitude
ax1.semilogx(w_out, mag_dB, 'b-', linewidth=2, label='G(s)')
ax1.axhline(0, color='r', linestyle='--', alpha=0.7, linewidth=1.5, label='0 dB')
if wcg:
    ax1.axvline(wcg, color='g', linestyle=':', alpha=0.8, label=f'ωcg = {wcg:.3f} r/s')
    ax1.annotate(f'  ωcg = {wcg:.2f}', xy=(wcg, 0), color='g', fontsize=9)
ax1.set_ylabel('Magnitude (dB)')
ax1.legend(loc='upper right', fontsize=10)

# Phase
ax2.semilogx(w_out, phase_deg, 'b-', linewidth=2, label='G(s)')
ax2.axhline(-180, color='r', linestyle='--', alpha=0.7, linewidth=1.5, label='-180°')
if wcp:
    ax2.axvline(wcp, color='orange', linestyle=':', alpha=0.9, label=f'ωcp = {wcp:.3f} r/s')
    ax2.annotate(f'  ωcp = {wcp:.2f}', xy=(wcp, -200), color='orange', fontsize=9)
if PM and wcg:
    ax2.annotate('', xy=(wcg, -180), xytext=(wcg, -180+PM),
                 arrowprops=dict(arrowstyle='<->', color='green', lw=1.5))
    ax2.text(wcg*1.1, -180+PM/2, f'PM={PM:.1f}°', color='green', fontsize=9)
ax2.set_ylabel('Phase (°)')
ax2.set_xlabel('ω (rad/s)')
ax2.legend(loc='lower left', fontsize=10)

plt.tight_layout()
plt.show()

## 3. Respon Step Baseline (CL Tanpa Kontroler)

Closed-loop dengan unity feedback dan tanpa kontroler:

$$T(s) = \frac{G(s)}{1 + G(s)} = \frac{8}{s^3 + 9s^2 + 20s + 20}$$

In [ ]:
# Closed-loop tanpa kontroler
num_CL0 = num_G
den_CL0 = np.polyadd(den_G, num_G)   # den + num = 1 + G numerator → CL denominator

CL0 = signal.TransferFunction(num_CL0, den_CL0)

# Step response
t_sim = np.linspace(0, 20, 2000)
t0, y0 = signal.step(CL0, T=t_sim)

# Metrics
yss0 = y0[-1]
ymax0 = np.max(y0)
OS0 = 100 * (ymax0 - yss0) / yss0 if yss0 > 0.01 else 0
band = 0.02 * abs(yss0)
ts0 = t0[np.where(np.abs(y0 - yss0) > band)[0][-1] + 1] if np.any(np.abs(y0-yss0) > band) else 0

print('Respon Step — CL tanpa kontroler')
print('-' * 40)
print(f'  yss        = {yss0:.4f}  (steady-state, bukan 1 karena Type 0!)')
print(f'  ess (step) = {1 - yss0:.4f} = {100*(1-yss0):.2f}%')
print(f'  Overshoot  = {OS0:.2f}%  → spec OS ≤ 20%: {"✓" if OS0 <= 20 else "✗"}')
print(f'  ts (2%)    = {ts0:.3f} s → spec ts ≤ 4 s: {"✓" if ts0 <= 4 else "✗"}')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t0, y0, 'b-', linewidth=2, label='CL tanpa kontroler')
ax.axhline(1, color='k', linestyle='--', alpha=0.5, linewidth=1, label='Referensi = 1')
ax.axhline(yss0, color='gray', linestyle=':', alpha=0.7, linewidth=1, label=f'yss = {yss0:.3f}')
ax.axhline(yss0*(1+0.02), color='g', linestyle=':', alpha=0.5, linewidth=1)
ax.axhline(yss0*(1-0.02), color='g', linestyle=':', alpha=0.5, linewidth=1, label='±2% band')
ax.axvline(ts0, color='orange', linestyle=':', alpha=0.8, label=f'ts = {ts0:.2f} s')
ax.set_xlabel('t (s)')
ax.set_ylabel('y(t)')
ax.set_title('Step response — baseline (tanpa kontroler)')
ax.legend(fontsize=10)
ax.text(ts0+0.3, 0.05, f'OS = {OS0:.1f}%', color='red', fontsize=10)
plt.tight_layout()
plt.show()

## 4. Desain PID — Metode Ziegler-Nichols (Metode 2: Frekuensi)

**Prosedur ZN Metode 2:**
1. Tingkatkan gain K secara bertahap hingga sistem osilasi berkelanjutan
2. Gain pada kondisi tersebut = **Ku** (ultimate gain)
3. Periode osilasi = **Tu** (ultimate period)
4. Secara numerik: cari ωpc di mana ∠G(jωpc) = −180°, lalu Ku = 1/|G(jωpc)| dan Tu = 2π/ωpc

**Rumus ZN untuk PID:**

| Parameter | Nilai |
|-----------|-------|
| Kp | 0.6 · Ku |
| Ti (integral time) | 0.5 · Tu |
| Td (derivative time) | 0.125 · Tu |
| Ki = Kp/Ti | 1.2 · Ku / Tu |
| Kd = Kp·Td | 0.075 · Ku · Tu |

In [ ]:
# ---- ZN Method 2: Cari phase crossover ----
w_fine = np.logspace(-3, 5, 50000)
_, mag_dB_fine, phase_fine = signal.bode(G, w=w_fine)

# Cari crossing phase = -180°
cross180 = np.where(np.diff(np.sign(phase_fine + 180)))[0]

wpc = None
for idx in cross180:
    if phase_fine[idx] < -90:   # pastikan downward crossing (fase menurun melewati -180)
        t = -(phase_fine[idx] + 180) / (phase_fine[idx+1] - phase_fine[idx])
        wpc = np.exp(np.log(w_fine[idx]) + t*(np.log(w_fine[idx+1]) - np.log(w_fine[idx])))
        mag_at_wpc = mag_dB_fine[idx] + t*(mag_dB_fine[idx+1] - mag_dB_fine[idx])
        break

if wpc is not None:
    Ku = 10**(-mag_at_wpc / 20)  # Ku = 1/|G(jωpc)|
    Tu = 2 * np.pi / wpc

    # ZN PID parameters
    Kp_pid = 0.6 * Ku
    Ti_pid = 0.5 * Tu
    Td_pid = 0.125 * Tu
    Ki_pid = Kp_pid / Ti_pid
    Kd_pid = Kp_pid * Td_pid

    print('Hasil ZN Metode 2')
    print('=' * 45)
    print(f'  Phase crossover freq. ωpc = {wpc:.4f} rad/s')
    print(f'  |G(jωpc)| = {10**(mag_at_wpc/20):.5f} (= {mag_at_wpc:.2f} dB)')
    print(f'  Ku = 1/|G(jωpc)|          = {Ku:.4f}')
    print(f'  Tu = 2π / ωpc             = {Tu:.4f} s')
    print()
    print('  Parameter PID (ZN):')
    print(f'    Kp = 0.6 × Ku            = {Kp_pid:.4f}')
    print(f'    Ti = 0.5 × Tu            = {Ti_pid:.4f} s')
    print(f'    Td = 0.125 × Tu          = {Td_pid:.4f} s')
    print(f'    Ki = Kp/Ti               = {Ki_pid:.4f}')
    print(f'    Kd = Kp×Td               = {Kd_pid:.4f}')
    print()
    print(f'  C(s) = {Kp_pid:.4f} + {Ki_pid:.4f}/s + {Kd_pid:.4f}·s')
    print(f'       = ({Kd_pid:.4f}s² + {Kp_pid:.4f}s + {Ki_pid:.4f}) / s')
else:
    print('Phase crossover tidak ditemukan — coba ZN Metode 1 (step response)')

## 5. Analisis Closed-Loop dengan PID

PID controller dalam domain s:

$$C_{PID}(s) = K_p + \frac{K_i}{s} + K_d s = \frac{K_d s^2 + K_p s + K_i}{s}$$

Loop gain: $L(s) = C(s) \cdot G(s)$

Closed-loop: $T(s) = \frac{L(s)}{1 + L(s)}$

In [ ]:
# ---- PID TF ----
num_C = [Kd_pid, Kp_pid, Ki_pid]   # Kd·s² + Kp·s + Ki
den_C = [1, 0]                      # s

# Loop gain: C(s)·G(s)
num_L_pid = np.polymul(num_C, num_G)
den_L_pid = np.polymul(den_C, den_G)

# Closed-loop: L/(1+L) = num_L / (den_L + num_L)
num_CL_pid = num_L_pid
den_CL_pid = np.polyadd(den_L_pid, num_L_pid)

CL_pid = signal.TransferFunction(num_CL_pid, den_CL_pid)
L_pid  = signal.TransferFunction(num_L_pid, den_L_pid)

# Margins dari loop gain
PM_pid, GM_pid, wcg_pid, wcp_pid = compute_margins(L_pid)

# Step response
t_sim2 = np.linspace(0, 20, 2000)
t_pid, y_pid = signal.step(CL_pid, T=t_sim2)

# Metrics
yss_pid = np.mean(y_pid[-200:])
ymax_pid = np.max(y_pid)
OS_pid = 100*(ymax_pid - yss_pid)/yss_pid if yss_pid > 0.01 else 0
band_pid = 0.02*abs(yss_pid)
exceed = np.where(np.abs(y_pid - yss_pid) > band_pid)[0]
ts_pid = t_pid[exceed[-1]+1] if len(exceed) > 0 else 0

print('Margin Stabilitas — Loop Gain C_PID × G')
print('-' * 45)
print(f'  PM  = {PM_pid:.2f}°  (target: tidak ada, tapi ≥ 30° direkomendasikan)')
print(f'  GM  = {GM_pid:.2f} dB  (spec ≥ 10 dB: {"✓" if GM_pid >= 10 else "✗"})')
print()
print('Respon Step — CL dengan PID')
print('-' * 45)
print(f'  yss        = {yss_pid:.4f}')
print(f'  Overshoot  = {OS_pid:.2f}%   → spec ≤ 20%: {"✓" if OS_pid <= 20 else "✗"}')
print(f'  ts (2%)    = {ts_pid:.3f} s  → spec ≤ 4 s: {"✓" if ts_pid <= 4 else "✗"}')

# Visualisasi
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Step response
ax = axes[0]
ax.plot(t0, y0, color='#9CA3AF', linewidth=1.5, linestyle='--', label='Tanpa kontroler')
ax.plot(t_pid, y_pid, 'b-', linewidth=2.5, label='Dengan PID (ZN)')
ax.axhline(1, color='k', linestyle='--', alpha=0.4, linewidth=1, label='Referensi = 1')
ax.axhline(yss_pid*(1.02), color='g', linestyle=':', alpha=0.5, linewidth=1)
ax.axhline(yss_pid*(0.98), color='g', linestyle=':', alpha=0.5, linewidth=1, label='±2% band')
ax.axvline(ts_pid, color='orange', linestyle=':', alpha=0.8, label=f'ts = {ts_pid:.2f} s')
ax.set_xlabel('t (s)'); ax.set_ylabel('y(t)')
ax.set_title('Step response — CL dengan PID')
ax.legend(fontsize=9)
ax.set_ylim(-0.1, max(1.5, ymax_pid*1.1))

# Bode loop gain
w = np.logspace(-2, 3, 500)
_, mg0, ph0 = signal.bode(G, w=w)
_, mg_pid, ph_pid = signal.bode(L_pid, w=w)

ax2 = axes[1]
ax2.semilogx(w, mg0, color='#9CA3AF', linewidth=1.5, linestyle='--', label='G(s)')
ax2.semilogx(w, mg_pid, 'b-', linewidth=2, label='C_PID · G(s)')
ax2.axhline(0, color='r', linestyle='--', alpha=0.7, linewidth=1.5, label='0 dB')
if wcg_pid: ax2.axvline(wcg_pid, color='g', linestyle=':', alpha=0.7)
ax2.set_xlabel('ω (rad/s)'); ax2.set_ylabel('Magnitude (dB)')
ax2.set_title('Bode magnitude — loop gain dengan PID')
ax2.legend(fontsize=9)

plt.suptitle('Analisis PID (Ziegler-Nichols)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Desain Kompensator Lead — Metode Bode

**Tujuan:** Meningkatkan Phase Margin hingga ≥ 45° (target) untuk memperbaiki stabilitas dan overshoot.

**Fungsi alih kompensator Lead:**
$$G_c(s) = \frac{\alpha T s + 1}{T s + 1}, \quad \alpha > 1$$

**Prosedur desain (langkah demi langkah):**
1. Tentukan PM yang dibutuhkan: $\phi_{add} = PM_{target} - PM_{current} + \phi_{safety}$
2. Hitung α dari: $\sin(\phi_{max}) = \frac{\alpha - 1}{\alpha + 1}$
3. Tentukan ωm baru: cari ω di mana $|G(j\omega)|_{dB} = -10\log_{10}(\alpha)/2$
4. Hitung T: $T = \frac{1}{\omega_m \sqrt{\alpha}}$

In [ ]:
# ---- Desain Lead Compensator ----
PM_target = 50      # target phase margin (°)
safety    = 8       # safety margin tambahan (°)

phi_add_deg = PM_target - PM + safety
phi_add_rad = np.deg2rad(phi_add_deg)

print(f'PM saat ini (OL)    = {PM:.2f}°')
print(f'Target PM           = {PM_target}°')
print(f'Safety margin       = {safety}°')
print(f'Tambahan fase (φ_add) = {phi_add_deg:.2f}°')
print()

# Hitung alpha
sin_phi = np.sin(phi_add_rad)
alpha = (1 + sin_phi) / (1 - sin_phi)
print(f'sin(φ_add) = {sin_phi:.4f}')
print(f'α = (1 + sin(φ)) / (1 - sin(φ)) = {alpha:.4f}')
print()

# Cari ωm: frekuensi di mana |G(jω)|_dB = -10·log10(α)/2
target_mag_dB = -10 * np.log10(alpha) / 2
print(f'Target magnitude: |G(jωm)| = 1/√α = {1/np.sqrt(alpha):.4f}')
print(f'  dalam dB: -10·log10(α)/2 = {target_mag_dB:.4f} dB')
print()

w_dense = np.logspace(-3, 5, 100000)
_, mag_d, ph_d = signal.bode(G, w=w_dense)

idx_wm = np.where(np.diff(np.sign(mag_d - target_mag_dB)))[0]
wm = None
for idx in idx_wm:
    t = (target_mag_dB - mag_d[idx]) / (mag_d[idx+1] - mag_d[idx])
    wm = np.exp(np.log(w_dense[idx]) + t*(np.log(w_dense[idx+1]) - np.log(w_dense[idx])))
    break

if wm is not None:
    T_lead = 1 / (wm * np.sqrt(alpha))
    z_lead = 1 / (alpha * T_lead)   # zero at s = -1/(αT)
    p_lead = 1 / T_lead             # pole at s = -1/T

    print(f'ωm (new crossover)  = {wm:.4f} rad/s')
    print(f'T = 1/(ωm·√α)       = {T_lead:.6f} s')
    print()
    print('Kompensator Lead:')
    print(f'  Gc(s) = (αTs+1)/(Ts+1)')
    print(f'        = ({alpha*T_lead:.5f}s + 1) / ({T_lead:.5f}s + 1)')
    print(f'  Zero  : s = -1/(αT) = -{z_lead:.4f} rad/s')
    print(f'  Pole  : s = -1/T    = -{p_lead:.4f} rad/s')
    print(f'  Maksimum fase lead  = {np.rad2deg(np.arcsin((alpha-1)/(alpha+1))):.2f}°  (pada ω = {wm:.4f} r/s)')

## 7. Analisis Closed-Loop dengan Lead Compensator

In [ ]:
# ---- Lead TF ----
num_lead = [alpha * T_lead, 1]   # αTs + 1
den_lead = [T_lead, 1]           # Ts  + 1

# Loop gain: Lead × G
num_L_lead = np.polymul(num_lead, num_G)
den_L_lead = np.polymul(den_lead, den_G)

# Closed-loop
num_CL_lead = num_L_lead
den_CL_lead = np.polyadd(den_L_lead, num_L_lead)

CL_lead = signal.TransferFunction(num_CL_lead, den_CL_lead)
L_lead  = signal.TransferFunction(num_L_lead,  den_L_lead)

# Margins
PM_lead, GM_lead, wcg_lead, wcp_lead = compute_margins(L_lead)

# Step response
t_lead, y_lead = signal.step(CL_lead, T=t_sim2)

# Metrics
yss_lead = np.mean(y_lead[-200:])
ymax_lead = np.max(y_lead)
OS_lead = 100*(ymax_lead - yss_lead)/yss_lead if yss_lead > 0.01 else 0
band_lead = 0.02*abs(yss_lead)
exceed_lead = np.where(np.abs(y_lead - yss_lead) > band_lead)[0]
ts_lead = t_lead[exceed_lead[-1]+1] if len(exceed_lead) > 0 else 0

print('Margin Stabilitas — Loop Gain Lead × G')
print('-' * 45)
print(f'  PM  = {PM_lead:.2f}°   (target ≥ 50°: {"✓" if PM_lead >= 50 else "✗"})')
print(f'  GM  = {GM_lead:.2f} dB  (spec ≥ 10 dB: {"✓" if GM_lead >= 10 else "✗"})')
print()
print('Respon Step — CL dengan Lead Compensator')
print('-' * 45)
print(f'  yss        = {yss_lead:.4f}')
print(f'  Overshoot  = {OS_lead:.2f}%   → spec ≤ 20%: {"✓" if OS_lead <= 20 else "✗"}')
print(f'  ts (2%)    = {ts_lead:.3f} s  → spec ≤ 4 s: {"✓" if ts_lead <= 4 else "✗"}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(t0, y0, color='#9CA3AF', linewidth=1.5, linestyle='--', label='Tanpa kontroler')
ax.plot(t_lead, y_lead, 'b-', linewidth=2.5, label='Dengan Lead Gc(s)')
ax.axhline(1, color='k', linestyle='--', alpha=0.4, linewidth=1)
ax.axhline(yss_lead*(1.02), color='g', linestyle=':', alpha=0.5, linewidth=1)
ax.axhline(yss_lead*(0.98), color='g', linestyle=':', alpha=0.5, linewidth=1, label='±2% band')
ax.axvline(ts_lead, color='orange', linestyle=':', alpha=0.8, label=f'ts = {ts_lead:.2f} s')
ax.set_xlabel('t (s)'); ax.set_ylabel('y(t)')
ax.set_title('Step response — CL dengan Lead')
ax.legend(fontsize=9)

# Bode comparison
w_bode = np.logspace(-2, 3, 500)
_, mg_G,  ph_G  = signal.bode(G,      w=w_bode)
_, mg_Ll, ph_Ll = signal.bode(L_lead, w=w_bode)

ax2 = axes[1]
ax2.semilogx(w_bode, ph_G,  color='#9CA3AF', linewidth=1.5, linestyle='--', label='G(s)')
ax2.semilogx(w_bode, ph_Ll, 'b-', linewidth=2, label='Gc_lead · G(s)')
ax2.axhline(-180, color='r', linestyle='--', alpha=0.7, linewidth=1.5, label='-180°')
if wcg_lead:
    ph_at_wcg_lead = np.interp(wcg_lead, w_bode, ph_Ll)
    ax2.annotate('', xy=(wcg_lead, -180), xytext=(wcg_lead, ph_at_wcg_lead),
                 arrowprops=dict(arrowstyle='<->', color='green', lw=1.5))
    ax2.text(wcg_lead*1.1, (-180+ph_at_wcg_lead)/2, f'PM={PM_lead:.1f}°', color='green', fontsize=9)
ax2.set_xlabel('ω (rad/s)'); ax2.set_ylabel('Phase (°)')
ax2.set_title('Bode phase — sebelum & sesudah Lead')
ax2.legend(fontsize=9)

plt.suptitle('Analisis Lead Compensator', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Perbandingan Lengkap

Visualisasikan ketiga kondisi (tanpa kontroler, PID, Lead) dalam satu figure.

In [ ]:
w_cmp = np.logspace(-2, 3, 500)
_, mg_G,  ph_G  = signal.bode(G,     w=w_cmp)
_, mg_Lp, ph_Lp = signal.bode(L_pid, w=w_cmp)
_, mg_Ll, ph_Ll = signal.bode(L_lead,w=w_cmp)

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

colors = {'base': '#9CA3AF', 'pid': '#EF4444', 'lead': '#2563EB'}

# ---- Step response ----
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(t0,     y0,     color=colors['base'], linewidth=1.5, linestyle='--', label='Tanpa kontroler')
ax1.plot(t_pid,  y_pid,  color=colors['pid'],  linewidth=2,   label='PID (ZN)')
ax1.plot(t_lead, y_lead, color=colors['lead'], linewidth=2,   label='Lead Gc(s)')
ax1.axhline(1, color='k', linestyle='--', alpha=0.35, linewidth=1)
ax1.set_xlabel('t (s)'); ax1.set_ylabel('y(t)')
ax1.set_title('Step response — perbandingan')
ax1.legend(fontsize=9); ax1.set_xlim(0, 15)

# ---- Bode magnitude ----
ax2 = fig.add_subplot(gs[0, 1])
ax2.semilogx(w_cmp, mg_G,  color=colors['base'], linewidth=1.5, linestyle='--', label='G(s)')
ax2.semilogx(w_cmp, mg_Lp, color=colors['pid'],  linewidth=2,   label='C_PID·G')
ax2.semilogx(w_cmp, mg_Ll, color=colors['lead'], linewidth=2,   label='C_lead·G')
ax2.axhline(0, color='k', linestyle='--', alpha=0.5, linewidth=1)
ax2.set_xlabel('ω (rad/s)'); ax2.set_ylabel('Magnitude (dB)')
ax2.set_title('Bode magnitude — loop gain')
ax2.legend(fontsize=9)

# ---- Bode phase ----
ax3 = fig.add_subplot(gs[1, 0])
ax3.semilogx(w_cmp, ph_G,  color=colors['base'], linewidth=1.5, linestyle='--', label='G(s)')
ax3.semilogx(w_cmp, ph_Lp, color=colors['pid'],  linewidth=2,   label='C_PID·G')
ax3.semilogx(w_cmp, ph_Ll, color=colors['lead'], linewidth=2,   label='C_lead·G')
ax3.axhline(-180, color='k', linestyle='--', alpha=0.5, linewidth=1)
ax3.set_xlabel('ω (rad/s)'); ax3.set_ylabel('Phase (°)')
ax3.set_title('Bode phase — loop gain')
ax3.legend(fontsize=9)

# ---- Tabel summary ----
ax4 = fig.add_subplot(gs[1, 1])
ax4.axis('off')
tbl_data = [
    ['Parameter',         'No ctrl',        'PID (ZN)',                  'Lead Gc'],
    ['yss',               f'{yss0:.3f}',     f'{yss_pid:.3f}',            f'{yss_lead:.3f}'],
    ['OS (%)',            f'{OS0:.1f}',       f'{OS_pid:.1f}',             f'{OS_lead:.1f}'],
    ['ts (s, 2%)',        f'{ts0:.2f}',       f'{ts_pid:.2f}',             f'{ts_lead:.2f}'],
    ['PM (°)',            f'{PM:.1f}',        f'{PM_pid:.1f}',             f'{PM_lead:.1f}'],
    ['GM (dB)',           f'{GM:.1f}',        f'{GM_pid:.1f}',             f'{GM_lead:.1f}'],
    ['GM ≥ 10 dB?',       'No',  f'{"Yes" if GM_pid>=10 else "No"}',    f'{"Yes" if GM_lead>=10 else "No"}'],
    ['OS ≤ 20%?',         f'{"Yes" if OS0<=20 else "No"}', f'{"Yes" if OS_pid<=20 else "No"}',   f'{"Yes" if OS_lead<=20 else "No"}'],
    ['ts ≤ 4 s?',         f'{"Yes" if ts0<=4 else "No"}',  f'{"Yes" if ts_pid<=4 else "No"}',    f'{"Yes" if ts_lead<=4 else "No"}'],
]
tbl = ax4.table(cellText=tbl_data[1:], colLabels=tbl_data[0],
                cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
for (r,c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#1E40AF'); cell.set_text_props(color='white', fontweight='bold')
    elif 'Yes' in cell.get_text().get_text():
        cell.set_facecolor('#DCFCE7')
    elif 'No' in cell.get_text().get_text() and r > 5:
        cell.set_facecolor('#FEE2E2')
ax4.set_title('Ringkasan hasil', fontsize=11, pad=10)

fig.suptitle('Perbandingan: Tanpa Kontroler vs PID vs Lead Compensator', fontsize=13)
plt.show()

## 9. Kesimpulan

Isi bagian ini secara mandiri berdasarkan hasil simulasi kelompok Anda.

**Template kesimpulan:**

1. **Analisis plant:** Plant G(s) merupakan sistem Tipe ___, orde ___, stabil open-loop (PM = ___°, GM = ___ dB). Spesifikasi kinerja yang belum terpenuhi adalah ___.

2. **Desain PID (ZN):** Diperoleh Ku = ___, Tu = ___ s. Parameter PID: Kp = ___, Ki = ___, Kd = ___. Setelah diterapkan, OS = ___%, ts = ___ s. Spesifikasi yang terpenuhi: ___. Kekurangan: ___.

3. **Desain Kompensator Lead:** Dibutuhkan tambahan fase ___° untuk mencapai PM target ___°. Diperoleh α = ___, T = ___ s. Setelah diterapkan, PM = ___°, OS = ___%, ts = ___ s.

4. **Perbandingan:** Pendekatan ___ memberikan hasil lebih baik karena ___. Rekomendasi untuk aplikasi sistem ini adalah ___.

---

**Referensi:**
- Ogata, K. (2010). *Modern Control Engineering*, 5th ed., Bab 10–11.
- Nise, N.S. (2019). *Control Systems Engineering*, 8th ed., Bab 9–11.